# Library

In [1]:
!pip install tensorflow transformers
!pip install nltk

In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import re
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from tensorflow import keras
from tensorflow.keras.layers import Dense,Dropout, Input
from tqdm import tqdm
import pickle
from sklearn.metrics import confusion_matrix,f1_score,classification_report
import matplotlib.pyplot as plt
import itertools
from sklearn.utils import shuffle
from tensorflow.keras import regularizers
from transformers import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
nltk.download('punkt')

2026-06-05 06:21:31.336723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780640491.359842     200 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780640491.367342     200 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780640491.385811     200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780640491.385834     200 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780640491.385836     200 computation_placer.cc:177] computation placer alr

True

In [3]:
from transformers import DistilBertModel, DistilBertTokenizer

In [4]:
from transformers import MobileBertTokenizer, MobileBertModel

# Prepare dataset

Split data menjadi perkalimat untuk mendapat hasil akhir kalimat & label

In [5]:
data = pd.read_csv('/kaggle/input/datasets/keysyaa26/summarize/file_summarize_clean.csv')
df = data [['text_cleaned', 'summary_cleaned']]

## Otomatization Labeling

In [6]:
class AutomaticLabeling:
    def __init__(self):
        pass

    def split_sentences(self, text):
        """
        Memecah paragraf menjadi list perkalimat
        """
        sentences = re.split(r'(?<=[.!?])\s+', str(text))
        
        return[
            sentence.strip() for sentence in sentences if sentence.strip()
        ]

    def calculate_similarity(self, sentences, summary):
        """
        Menghitung kemiripan kalimat dengan summary menggunakan TF-IDF
        """
        corpus = sentences + [summary]
        vectorize = TfidfVectorizer()
        vectors = vectorize.fit_transform(corpus)

        sentence_vector = vectors[:-1]
        summary_vector = vectors[-1:]

        similarities = cosine_similarity(sentence_vector, summary_vector).flatten()

        return similarities

    def labeling_sentence(self, document, summary):
        """
        Membuat pseudo-label untuk kalimat
        1 = penting
        0 = tidak penting
        """
        sentences = self.split_sentences(document)

        if len(sentences) == 0:
            return []

        similarities = self.calculate_similarity(sentences, summary)

        summary_sentences = self.split_sentences(summary)

        k = max(1, min(len(summary_sentences), len(sentences)))

        top_indices = np.argsort(similarities)[-k:]

        labeled_data = []

        for idx, sentence in enumerate(sentences):
            labeled_data.append({
                'sentence': sentence,
                'label': int(idx in top_indices)
            })

        return labeled_data

In [7]:
label_otomatis = AutomaticLabeling()

In [8]:
sample = df.iloc[0]
sample

text_cleaned       - Dokter Ryan Thamrin , yang terkenal lewat ac...
summary_cleaned    Dokter Lula Kamal yang merupakan selebriti sek...
Name: 0, dtype: object

In [9]:
result = label_otomatis.labeling_sentence(
    sample['text_cleaned'],
    sample['summary_cleaned']
)

In [10]:
for item in result:
    print(item)

{'sentence': '- Dokter Ryan Thamrin , yang terkenal lewat acara Dokter Oz Indonesia , meninggal dunia pada Jumat 4 8 dini hari .', 'label': 0}
{'sentence': 'Dokter Lula Kamal yang merupakan selebriti sekaligus rekan kerja Ryan menyebut kawannya itu sudah sakit sejak setahun yang lalu .', 'label': 1}
{'sentence': 'Lula menuturkan , sakit itu membuat Ryan mesti vakum dari semua kegiatannya , termasuk menjadi pembawa acara Dokter Oz Indonesia .', 'label': 1}
{'sentence': 'Kondisi itu membuat Ryan harus kembali ke kampung halamannya di Pekanbaru , Riau untuk menjalani istirahat .', 'label': 1}
{'sentence': '" Setahu saya dia orangnya sehat , tapi tahun lalu saya dengar dia sakit .', 'label': 0}
{'sentence': 'Karena sakitnya , ia langsung pulang ke Pekanbaru , jadi kami yang mau jenguk juga susah .', 'label': 0}
{'sentence': 'Barangkali mau istirahat , ya betul juga , kalau di Jakarta susah isirahatnya , " kata Lula kepada CNNIndonesia.com , Jumat 4 8 .', 'label': 0}
{'sentence': 'Lula yang

In [11]:
all_sentences = []

# labeling semua data
for _, row in tqdm(df.iterrows(),total=len(df)):
    labeled_data = label_otomatis.labeling_sentence(
        row["text_cleaned"],
        row["summary_cleaned"]
    )
    all_sentences.extend(labeled_data)

  0%|          | 0/11516 [00:00<?, ?it/s]

In [12]:
sentence_df = pd.DataFrame(all_sentences)

sentence_df.head()

,sentence,label
0,"- Dokter Ryan Thamrin , yang terkenal lewat ac...",0
1,Dokter Lula Kamal yang merupakan selebriti sek...,1
2,"Lula menuturkan , sakit itu membuat Ryan mesti...",1
3,Kondisi itu membuat Ryan harus kembali ke kamp...,1
4,""" Setahu saya dia orangnya sehat , tapi tahun ...",0


In [13]:
sentence_df["label"].value_counts()

label
0    145919
1     81506
Name: count, dtype: int64

## Split Dataset

In [14]:
train_df, temp_df = train_test_split(
    sentence_df,
    test_size=0.2,
    random_state=42,
    stratify=sentence_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

In [15]:
print("Train :", len(train_df))
print("Val   :", len(val_df))
print("Test  :", len(test_df))

Train : 181940
Val   : 22742
Test  : 22743


In [16]:
print(train_df["label"].value_counts(normalize=True))
print(val_df["label"].value_counts(normalize=True))
print(test_df["label"].value_counts(normalize=True))

label
0    0.641613
1    0.358387
Name: proportion, dtype: float64
label
0    0.641632
1    0.358368
Name: proportion, dtype: float64
label
0    0.641604
1    0.358396
Name: proportion, dtype: float64


# Model

In [17]:
# load DistillBERT Tokenizer
tokenizer = MobileBertTokenizer.from_pretrained("google/mobilebert-uncased")

In [18]:
# hyperparameter
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

In [19]:
print(MAX_LENGTH)

128


## Encoding & Transform to tensorflow dataset

In [20]:
# tokenisasi dataset
def tokenize_bert(text):
    return tokenizer(text, padding="max_length", truncation=True, max_length=128)

In [21]:
# encode
train_encode = tokenize_bert(train_df["sentence"].tolist())

In [22]:
train_labels = train_df["label"].values
val_labels = val_df["label"].values
test_labels = test_df["label"].values

Menghasilkan 3 parameter untuk input mobile bert, input_ids, token_type_ids, attention mask

In [23]:
test = tokenize_bert(train_df['sentence'][0])

In [24]:
train_labels

array([1, 0, 0, ..., 1, 0, 1])

In [25]:
print(len(train_encode["input_ids"][0]))
print(len(train_encode["input_ids"][1]))
print(len(train_encode["input_ids"][2]))

128
128
128


In [26]:
train_ds = tf.data.Dataset.from_tensor_slices((dict(train_encode), train_labels))

I0000 00:00:1780640582.111520     200 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780640582.117774     200 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [27]:
train_ds

<_TensorSliceDataset element_spec=({'input_ids': TensorSpec(shape=(128,), dtype=tf.int32, name=None), 'token_type_ids': TensorSpec(shape=(128,), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(128,), dtype=tf.int32, name=None)}, TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [28]:
val_encode = tokenize_bert(val_df["sentence"].tolist())
test_encode = tokenize_bert(test_df["sentence"].tolist())

In [29]:
val_ds = tf.data.Dataset.from_tensor_slices((dict(val_encode), val_labels))
test_ds = tf.data.Dataset.from_tensor_slices((dict(test_encode), test_labels))

In [30]:
val_ds

<_TensorSliceDataset element_spec=({'input_ids': TensorSpec(shape=(128,), dtype=tf.int32, name=None), 'token_type_ids': TensorSpec(shape=(128,), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(128,), dtype=tf.int32, name=None)}, TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [31]:
test_ds

<_TensorSliceDataset element_spec=({'input_ids': TensorSpec(shape=(128,), dtype=tf.int32, name=None), 'token_type_ids': TensorSpec(shape=(128,), dtype=tf.int32, name=None), 'attention_mask': TensorSpec(shape=(128,), dtype=tf.int32, name=None)}, TensorSpec(shape=(), dtype=tf.int64, name=None))>

## Batch Dataset

In [32]:
train_ds = (
    train_ds
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [33]:
for batch in train_ds.take(1):

    x, y = batch

    print(x["input_ids"].shape)
    print(x["attention_mask"].shape)
    print(y.shape)

(16, 128)
(16, 128)
(16,)


In [34]:
val_ds = (val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_ds = (test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [35]:
# dataset HF
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids": train_encode["input_ids"],
    "attention_mask": train_encode["attention_mask"],
    "token_type_ids": train_encode["token_type_ids"],
    "labels": train_labels
})

In [36]:
val_dataset = Dataset.from_dict({
    "input_ids": val_encode["input_ids"],
    "attention_mask": val_encode["attention_mask"],
    "token_type_ids": val_encode["token_type_ids"],
    "labels": val_labels
})

## Finetuning

In [37]:
from transformers import MobileBertForSequenceClassification

model = MobileBertForSequenceClassification.from_pretrained(
    "google/mobilebert-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--mobilebert-uncased/snapshots/1f90a6c24c7879273a291d34a849033eba2dbc0f/config.json
Model config MobileBertConfig {
  "architectures": [
    "MobileBertForPreTraining"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_activation": false,
  "classifier_dropout": null,
  "embedding_size": 128,
  "hidden_act": "relu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "intra_bottleneck_size": 128,
  "key_query_shared_bottleneck": true,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "mobilebert",
  "normalization_type": "no_norm",
  "num_attention_heads": 4,
  "num_feedforward_networks": 4,
  "num_hidden_layers": 24,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "trigram_input": true,
  "true_hidden_size": 128,
  "type_vocab_size": 2,
  "use_bottleneck": tr

pytorch_model.bin:   0%|          | 0.00/147M [00:00<?, ?B/s]

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--google--mobilebert-uncased/snapshots/1f90a6c24c7879273a291d34a849033eba2dbc0f/pytorch_model.bin
Attempting to create safetensors variant
Since the `dtype` attribute can't be found in model's config object, will use dtype={dtype} as derived from model's weights
Safetensors PR exists


model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
mobilebert.embeddings.position_ids         | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignore

In [38]:
model

MobileBertForSequenceClassification(
  (mobilebert): MobileBertModel(
    (embeddings): MobileBertEmbeddings(
      (word_embeddings): Embedding(30522, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (embedding_transformation): Linear(in_features=384, out_features=512, bias=True)
      (LayerNorm): NoNorm()
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): MobileBertEncoder(
      (layer): ModuleList(
        (0-23): 24 x MobileBertLayer(
          (attention): MobileBertAttention(
            (self): MobileBertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=512, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): MobileBertSelfOutput(
              (dense): Linear(in_fe

In [39]:
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

In [40]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./mobilebert_output",
    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100
)

PyTorch: setting up devices


In [41]:
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support

In [42]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary"
        )
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [43]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [44]:
trainer.train()

***** Running training *****
  Num examples = 181,940
  Num Epochs = 2
  Instantaneous batch size per device = 16
  Training with DataParallel so batch size has been adjusted to: 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 11,372
  Number of trainable parameters = 24,582,914
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.816221,0.831199,0.822707,0.877244,0.587485,0.703704
2,0.843077,0.796067,0.830270,0.864920,0.623804,0.724836



***** Running Evaluation *****
  Num examples = 22742
  Batch size = 32
Saving model checkpoint to ./mobilebert_output/checkpoint-5686
Configuration saved in ./mobilebert_output/checkpoint-5686/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./mobilebert_output/checkpoint-5686/model.safetensors
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]

***** Running Evaluation *****
  Num examples = 22742
  Batch size = 32
Saving model checkpoint to ./mobilebert_output/checkpoint-11372
Configuration saved in ./mobilebert_output/checkpoint-11372/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in ./mobilebert_output/checkpoint-11372/model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./mobilebert_output/checkpoint-11372 (score: 0.7248360422013117).
There were missing keys in the checkpoint model loaded: ['mobilebert.embeddings.LayerNorm.bias', 'mobilebert.embeddings.LayerNorm.weight', 'mobilebert.encoder.layer.0.attention.output.LayerNorm.bias', 'mobilebert.encoder.layer.0.attention.output.LayerNorm.weight', 'mobilebert.encoder.layer.0.output.LayerNorm.bias', 'mobilebert.encoder.layer.0.output.LayerNorm.weight', 'mobilebert.encoder.layer.0.output.bottleneck.LayerNorm.bias', 'mobilebert.encoder.layer.0.output.bottleneck.LayerNorm.weight', 'mobilebert.encoder.layer.0.bottleneck.input.LayerNorm.bias', 'mobilebert.encoder.layer.0.bottleneck.input.LayerNorm.weight', 'mobilebert.encoder.layer.0.bottleneck.attention.LayerNorm.bias', 'mobilebert.encoder.layer.0.bottleneck.attention.LayerN

TrainOutput(global_step=11372, training_loss=971.81440799269, metrics={'train_runtime': 3590.1793, 'train_samples_per_second': 101.354, 'train_steps_per_second': 3.168, 'total_flos': 5704594151731200.0, 'train_loss': 971.81440799269, 'epoch': 2.0})

In [45]:
trainer.save_model(
    "mobilebert_final"
)

tokenizer.save_pretrained(
    "mobilebert_final"
)

Saving model checkpoint to mobilebert_final
Configuration saved in mobilebert_final/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in mobilebert_final/model.safetensors
tokenizer config file saved in mobilebert_final/tokenizer_config.json


('mobilebert_final/tokenizer_config.json', 'mobilebert_final/tokenizer.json')

In [47]:
trainer.evaluate(
    val_dataset
)


***** Running Evaluation *****
  Num examples = 22742
  Batch size = 32
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.7960673570632935,
 'eval_accuracy': 0.8302699850496879,
 'eval_precision': 0.8649200408302143,
 'eval_recall': 0.6238036809815951,
 'eval_f1': 0.7248360422013117,
 'eval_runtime': 115.5009,
 'eval_samples_per_second': 196.899,
 'eval_steps_per_second': 6.156,
 'epoch': 2.0}

In [50]:
!pip install optimum[onnxruntime]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 100.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: transformers

In [51]:
!ls mobilebert_final

config.json	   tokenizer_config.json  training_args.bin
model.safetensors  tokenizer.json


In [53]:
!optimum-cli export onnx \
    --model /kaggle/working/mobilebert_output/checkpoint-5686 \
    --task text-classification \
    /kaggle/working/mobilebert_onnx

2026-06-05 07:32:19.776462: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780644739.800626   27384 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780644739.808105   27384 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780644739.828325   27384 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780644739.828369   27384 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780644739.828377   27384 computation_placer.cc:177] computation placer alr